# StockSense Pro / ShelfSmart AI — Full Pipeline (disk-optimized)

Same pipeline as before, but cleans up disk space as it goes: deletes the zip right after extraction, deletes the unused `train2019` folder immediately (you never use it — only `val2019`/`test2019` are used), and deletes the raw `rpc` folder once the YOLO dataset is built. This keeps peak disk usage much lower than before.

**Before running:** set Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**, then run cells in order.

## Step 1 — Kaggle setup and download

In [ ]:
!pip install -q kaggle

from google.colab import files
print("Upload your kaggle.json (Kaggle account -> Settings -> API -> Create New Token)")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d diyer22/retail-product-checkout-dataset -p /content/rpc_raw

## Step 2 — Extract, then immediately delete the zip and the unused train2019 folder

This is the main disk-saving change: the zip (~25GB) and `train2019` (53,739 images you never use) get removed right away instead of sitting around.

In [ ]:
import zipfile, os, shutil

zip_path = "/content/rpc_raw/retail-product-checkout-dataset.zip"
extract_path = "/content/rpc"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)

# Delete the zip immediately — fully extracted, no longer needed
shutil.rmtree("/content/rpc_raw")
print("Deleted zip / rpc_raw folder")

# The RPC zip nests everything under retail_product_checkout/ AND duplicates
# train2019/val2019/test2019 at the top level of /content/rpc — remove the
# top-level duplicates, keep the nested copies (that's what `base` points to below)
for folder in ["train2019", "val2019", "test2019"]:
    dup_path = f"/content/rpc/{folder}"
    if os.path.exists(dup_path):
        shutil.rmtree(dup_path)
for jf in ["instances_train2019.json", "instances_val2019.json", "instances_test2019.json"]:
    dup_json = f"/content/rpc/{jf}"
    if os.path.exists(dup_json):
        os.remove(dup_json)
print("Removed top-level duplicate folders/files")

# Delete train2019 inside retail_product_checkout/ too — we only use val2019 + test2019
train2019_path = "/content/rpc/retail_product_checkout/train2019"
if os.path.exists(train2019_path):
    shutil.rmtree(train2019_path)
    print("Deleted unused train2019 (53,739 images) — not used by this pipeline")

!df -h /content

## Step 3 — Confirm GPU is active

If this prints `False`, stop here and set Runtime -> Change runtime type -> T4 GPU, then re-run from Step 1 (the restart wipes `/content`).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## Step 4 — Load annotations (val2019 + test2019 only)

In [ ]:
import json
from collections import Counter

base = "/content/rpc/retail_product_checkout"

with open(f"{base}/instances_val2019.json") as f:
    val_data = json.load(f)
with open(f"{base}/instances_test2019.json") as f:
    test_data = json.load(f)

categories = {c["id"]: c["name"] for c in val_data["categories"]}
print("Total categories:", len(categories))

## Step 5 — Select 8 categories (balanced across 8 supercategories)

In [ ]:
SELECTED_CATEGORY_IDS = [97, 129, 173, 8, 67, 181, 82, 48]
# 97_milk, 129_chocolate, 173_personal_hygiene, 8_puffed_food,
# 67_dessert, 181_tissue, 82_drink, 48_instant_noodles

class_names = [categories[cid] for cid in SELECTED_CATEGORY_IDS]
cat_id_to_class_idx = {cid: i for i, cid in enumerate(SELECTED_CATEGORY_IDS)}
print("Total categories selected:", len(SELECTED_CATEGORY_IDS))
print("YOLO class order:", class_names)

selected_names = {cid: categories[cid] for cid in SELECTED_CATEGORY_IDS}
with open("/content/rpc_selected_categories.json", "w") as f:
    json.dump({str(cid): name for cid, name in selected_names.items()}, f, indent=2)
print("Saved category selection.")

from google.colab import files
files.download("/content/rpc_selected_categories.json")

## Step 6 — Build pool of images containing selected categories

In [ ]:
def collect_annotated_images(data, source_split):
    img_lookup = {img["id"]: img for img in data["images"]}
    result = {}
    for ann in data["annotations"]:
        if ann["category_id"] not in cat_id_to_class_idx:
            continue
        img_id = ann["image_id"]
        if img_id not in result:
            info = img_lookup[img_id]
            result[img_id] = {
                "file_name": info["file_name"],
                "width": info["width"],
                "height": info["height"],
                "source": source_split,
                "anns": []
            }
        result[img_id]["anns"].append(ann)
    return result

val_pool = collect_annotated_images(val_data, "val2019")
test_pool = collect_annotated_images(test_data, "test2019")

all_pool = {**val_pool, **test_pool}
print("Total images containing selected categories:", len(all_pool))

## Step 7 — Convert to YOLO format, resize to 640x640, split 70/15/15 (your own split, not RPC's original)

In [ ]:
import os, cv2, shutil
from tqdm import tqdm
import random

TARGET_SIZE = 640
OUT_ROOT = "/content/yolo_dataset"

if os.path.exists(OUT_ROOT):
    shutil.rmtree(OUT_ROOT)
os.makedirs(OUT_ROOT, exist_ok=True)

for split in ["train", "val", "test"]:
    os.makedirs(f"{OUT_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT_ROOT}/labels/{split}", exist_ok=True)

image_ids = list(all_pool.keys())
random.seed(42)
random.shuffle(image_ids)

n = len(image_ids)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

split_map = {}
for i, img_id in enumerate(image_ids):
    if i < n_train:
        split_map[img_id] = "train"
    elif i < n_train + n_val:
        split_map[img_id] = "val"
    else:
        split_map[img_id] = "test"

print("Split sizes:", {s: list(split_map.values()).count(s) for s in ["train","val","test"]})

skipped = 0
for img_id in tqdm(image_ids, desc="Processing images"):
    info = all_pool[img_id]
    split = split_map[img_id]
    src_path = f"{base}/{info['source']}/{info['file_name']}"
    if not os.path.exists(src_path):
        skipped += 1
        continue

    img = cv2.imread(src_path)
    if img is None:
        skipped += 1
        continue
    orig_h, orig_w = img.shape[:2]
    img_resized = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)

    stem = os.path.splitext(info["file_name"])[0]
    dst_img_path = f"{OUT_ROOT}/images/{split}/{stem}.jpg"
    cv2.imwrite(dst_img_path, img_resized, [cv2.IMWRITE_JPEG_QUALITY, 90])

    label_lines = []
    for ann in info["anns"]:
        x, y, w, h = ann["bbox"]
        x_center = (x + w / 2) / orig_w
        y_center = (y + h / 2) / orig_h
        w_norm = w / orig_w
        h_norm = h / orig_h
        cls_idx = cat_id_to_class_idx[ann["category_id"]]
        label_lines.append(f"{cls_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

    with open(f"{OUT_ROOT}/labels/{split}/{stem}.txt", "w") as f:
        f.write("\n".join(label_lines))

print("Done. Skipped (missing files):", skipped)
!df -h /content

## Step 8 — Write `data.yaml` for Ultralytics YOLO

In [ ]:
yaml_content = f"""path: {OUT_ROOT}
train: images/train
val: images/val
test: images/test

nc: {len(class_names)}
names: {class_names}
"""

with open(f"{OUT_ROOT}/data.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

## Step 9 — Delete the raw `rpc` folder now — yolo_dataset is self-contained

This is the biggest space reclaim. Training only reads from `/content/yolo_dataset` from here on, never from `/content/rpc` again.

In [ ]:
import shutil, os

if os.path.exists("/content/rpc"):
    shutil.rmtree("/content/rpc")
    print("Deleted raw rpc folder — yolo_dataset is self-contained now")

!df -h /content

## Step 10 — Sanity check: visualize one sample image with its converted YOLO labels

In [ ]:
import matplotlib.pyplot as plt
import os, random

sample_img_name = random.choice(os.listdir(f"{OUT_ROOT}/images/train"))
stem = os.path.splitext(sample_img_name)[0]

img = cv2.imread(f"{OUT_ROOT}/images/train/{sample_img_name}")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

with open(f"{OUT_ROOT}/labels/train/{stem}.txt") as f:
    lines = f.readlines()

fig, ax = plt.subplots(1, figsize=(6,6))
ax.imshow(img)
for line in lines:
    cls_idx, xc, yc, bw, bh = map(float, line.split())
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    box_w = bw * w
    box_h = bh * h
    rect = plt.Rectangle((x1, y1), box_w, box_h, fill=False, edgecolor='lime', linewidth=2)
    ax.add_patch(rect)
    ax.text(x1, y1-5, class_names[int(cls_idx)], color='lime', fontsize=9)
plt.title(sample_img_name)
plt.show()

## Step 11 — Install Ultralytics

In [ ]:
!pip install -q ultralytics

## Step 12 — Train YOLOv8 (nano model, fits Colab free-tier GPU)

~30-45 min expected on a T4 GPU for 30 epochs at this dataset size. `patience=10` stops early if validation performance plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{OUT_ROOT}/data.yaml",
    epochs=30,
    batch=16,
    imgsz=640,
    project="/content/runs",
    name="shelfsmart_yolo",
    patience=10
)

## Step 13 — Evaluate on the test set

In [ ]:
metrics = model.val(data=f"{OUT_ROOT}/data.yaml", split="test")
print(metrics)

## Step 14 — Download the trained model weights

You'll need `best.pt` for both your GitHub repo and your Streamlit app.

In [ ]:
from google.colab import files
best_weights_path = "/content/runs/shelfsmart_yolo/weights/best.pt"
print("Weights saved at:", best_weights_path)
files.download(best_weights_path)